# Job Training Programme : Data and Randomisation
### Synthetic data for a simulated policy evaluation. Nothing here is evidence about a real Indian government programme.

In [1]:
import os
import sys
import urllib.request
import pandas as pd
import plotly.express as px

sys.path.append('../..')
from self_module import diff_in_means

In [2]:
import plotly.io as pio
pio.templates["plotly_dark"].layout.paper_bgcolor = "black"
pio.templates["plotly_dark"].layout.plot_bgcolor = "black"
pio.templates.default = "plotly_dark"

## Business Understanding
### A state skills mission offered subsidised training to applicants at random. Not everyone offered enrolled , and a few not offered enrolled anyway.
### The government wants to know : the effect of offering training , the effect of the training itself , and whether OBC , SC and ST applicants benefit.
### Outcomes : employment and earnings at 18 and 30 months.

### The brief says 10,000 applicants in 16 districts , but the file it links to has 40,000 applicants in 45 districts. The linked file is used.

In [3]:
# download once , then read from disk
url = 'https://raw.githubusercontent.com/hirdeshiitkgp/Data/refs/heads/main/jtpa_india_synthetic_40000.csv'
path = '../../data/jtpa_india_synthetic_40000.csv'
if not os.path.exists(path):
    os.makedirs('../../data' , exist_ok=True)
    urllib.request.urlretrieve(url , path)

df = pd.read_csv(path)
df.head()

,participant_id,state,district,area_type,target_group,gender,age_years,social_category,disadvantaged_community_obc_sc_st,sc_st_flag,...,employed_prior_12mo,prior_12mo_earnings_inr,assigned_to_training,enrolled_in_training,training_type,months_in_training,employed_18mo,earnings_18mo_inr,employed_30mo,earnings_30mo_inr
0,IN300000,Uttar Pradesh,Bahraich,Urban,Youth (18-24),Male,20,OBC,1,0,...,1,93800.0,0,0,NaN,0.0,1,163700.0,0,0.0
1,IN300001,Rajasthan,Dholpur,Rural,Adult (25-45),Female,33,General,0,0,...,0,0.0,0,0,NaN,0.0,1,105200.0,1,164900.0
2,IN300002,Uttar Pradesh,Bahraich,Rural,Adult (25-45),Female,30,OBC,1,0,...,1,31700.0,1,1,Placement & job-search support,2.6,1,164900.0,1,15800.0
3,IN300003,Jharkhand,West Singhbhum,Rural,Youth (18-24),Male,21,General,0,0,...,1,83600.0,0,0,NaN,0.0,0,0.0,0,0.0
4,IN300004,Manipur,Churachandpur,Urban,Adult (25-45),Female,25,General,0,0,...,0,0.0,0,0,NaN,0.0,0,0.0,0,0.0


## 1. Data Understanding and Randomisation

In [4]:
df.shape

(40000, 25)

In [5]:
df.dtypes

participant_id                        object
state                                 object
district                              object
area_type                             object
target_group                          object
gender                                object
age_years                              int64
social_category                       object
disadvantaged_community_obc_sc_st      int64
sc_st_flag                             int64
years_of_schooling                     int64
completed_class10                      int64
married                                int64
num_dependents                         int64
bpl_card_holder                        int64
employed_prior_12mo                    int64
prior_12mo_earnings_inr              float64
assigned_to_training                   int64
enrolled_in_training                   int64
training_type                         object
months_in_training                   float64
employed_18mo                          int64
earnings_1

### 40,000 applicants and 25 columns.
| role | columns |
|---|---|
| assignment | assigned_to_training (the random offer) |
| treatment received | enrolled_in_training , training_type , months_in_training |
| outcomes | employed_18mo , earnings_18mo_inr , employed_30mo , earnings_30mo_inr |
| baseline | prior_12mo_earnings_inr , employed_prior_12mo , years_of_schooling , age_years , married , num_dependents |
| social groups | social_category , gender , area_type , district , target_group |

### Missing values

In [6]:
df.isna().sum()[df.isna().sum() > 0]

training_type    23140
dtype: int64

In [7]:
pd.crosstab(df['training_type'].fillna('None') , df['enrolled_in_training'])

enrolled_in_training,0,1
training_type,,
Classroom skill training,0,6018
None,23140,0
On-the-job training / apprenticeship,0,4742
Other support services,0,1983
Placement & job-search support,0,4117


### training_type is empty for exactly the 23,140 people who did not enrol. It means "no training" , not missing data.
### training_type and months_in_training happen after the offer , so they are never used as controls.

In [8]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age_years,40000.0,26.60,6.20,18.0,21.0,25.0,32.0,45.0
disadvantaged_community_obc_sc_st,40000.0,0.50,0.50,0.0,0.0,0.5,1.0,1.0
sc_st_flag,40000.0,0.28,0.45,0.0,0.0,0.0,1.0,1.0
years_of_schooling,40000.0,8.33,3.03,0.0,6.0,8.0,10.0,16.0
completed_class10,40000.0,0.35,0.48,0.0,0.0,0.0,1.0,1.0
married,40000.0,0.52,0.50,0.0,0.0,1.0,1.0,1.0
num_dependents,40000.0,1.57,1.44,0.0,0.0,1.0,2.0,10.0
bpl_card_holder,40000.0,0.32,0.46,0.0,0.0,0.0,1.0,1.0
employed_prior_12mo,40000.0,0.44,0.50,0.0,0.0,0.0,1.0,1.0
prior_12mo_earnings_inr,40000.0,33799.97,48846.58,0.0,0.0,0.0,60400.0,451700.0


In [9]:
(df[['earnings_18mo_inr' , 'earnings_30mo_inr']] == 0).mean()

earnings_18mo_inr    0.54790
earnings_30mo_inr    0.54385
dtype: float64

In [10]:
pd.crosstab(df['employed_30mo'] , df['earnings_30mo_inr'] > 0)

earnings_30mo_inr,False,True
employed_30mo,,
0,21754,0
1,0,18246


### About 54% earn nothing at 18 and 30 months. Earnings are 0 exactly when the person is not employed.

In [11]:
counts = df['social_category'].value_counts()
fig = px.bar(counts , title='social_category')
fig.show()
df['social_category'].value_counts(normalize=True)

social_category
General    0.50
OBC        0.22
SC         0.18
ST         0.10
Name: proportion, dtype: float64

### 50% General , 22% OBC , 18% SC , 10% ST.

### Balance : baseline characteristics of the assigned vs not assigned
### If the offer was random , the two groups should only differ by chance.

In [12]:
baseline = df[['prior_12mo_earnings_inr' , 'employed_prior_12mo' , 'years_of_schooling' , 'completed_class10' ,
               'age_years' , 'married' , 'num_dependents' , 'bpl_card_holder']].copy()
baseline['female'] = (df['gender'] == 'Female').astype(int)
baseline['rural'] = (df['area_type'] == 'Rural').astype(int)
baseline['youth'] = (df['target_group'] == 'Youth (18-24)').astype(int)
for g in ['OBC' , 'SC' , 'ST']:
    baseline[g] = (df['social_category'] == g).astype(int)

balance = pd.DataFrame({col : diff_in_means(baseline[col].values , df['assigned_to_training'].values) for col in baseline}).T
balance['diff / sd'] = balance['diff'] / baseline.std()
balance[['treated mean' , 'control mean' , 'diff' , 'diff / sd' , 'p']].round(3)

,treated mean,control mean,diff,diff / sd,p
prior_12mo_earnings_inr,33790.933,33818.053,-27.120,-0.001,0.958
employed_prior_12mo,0.437,0.434,0.002,0.005,0.666
years_of_schooling,8.324,8.344,-0.020,-0.007,0.537
completed_class10,0.350,0.350,-0.000,-0.001,0.927
age_years,26.602,26.586,0.016,0.003,0.814
married,0.523,0.517,0.005,0.011,0.320
num_dependents,1.573,1.575,-0.002,-0.001,0.916
bpl_card_holder,0.316,0.313,0.004,0.008,0.466
female,0.453,0.446,0.008,0.015,0.148
rural,0.728,0.721,0.008,0.017,0.101


### Every difference is below 0.03 standard deviations (0.1 is the usual line for a real imbalance).
### One p value out of 14 is below 0.05 : SC share (p = 0.017). With 14 tests about 0.7 are expected below 0.05 by chance , and the gap is 1 point.

### Assignment by state

In [13]:
rate = df.groupby('state')['assigned_to_training'].mean()
fig = px.bar(rate , title='Share assigned to training by state')
fig.show()
rate.round(4)

state
Assam             0.6668
Bihar             0.6666
Chhattisgarh      0.6668
Jharkhand         0.6666
Madhya Pradesh    0.6667
Manipur           0.6668
Meghalaya         0.6668
Odisha            0.6668
Rajasthan         0.6667
Tripura           0.6665
Uttar Pradesh     0.6666
West Bengal       0.6667
Name: assigned_to_training, dtype: float64

### Exactly 2 / 3 assigned in every state , so the offer was randomised within each state.
### Randomisation looks successful.

### Assignment and enrolment (2 x 2)

In [14]:
pd.crosstab(df['assigned_to_training'] , df['enrolled_in_training'] , margins=True)

enrolled_in_training,0,1,All
assigned_to_training,,,
0,12927,406,13333
1,10213,16454,26667
All,23140,16860,40000


In [15]:
df.groupby('assigned_to_training')['enrolled_in_training'].mean()

assigned_to_training
0    0.030451
1    0.617017
Name: enrolled_in_training, dtype: float64

### First stage : how much the offer raised enrolment

In [16]:
pd.Series(diff_in_means(df['enrolled_in_training'].values , df['assigned_to_training'].values)).round(4)

treated mean    0.6170
control mean    0.0305
diff            0.5866
se              0.0033
ci low          0.5800
ci high         0.5931
p               0.0000
dtype: float64

### 61.7% of those offered enrolled , 3.0% of those not offered enrolled anyway. The offer raised enrolment by 58.7 points (95% CI 58.0 to 59.3).
### Noncompliance : 38% of those offered never enrolled , and 406 people enrolled without the offer.
### So assigned and trained are not the same people. Comparing enrolled vs not enrolled would mix the training with who chooses to enrol.